In [1]:
# =====================================================================
# 🔮 NVDA 매일 예측 전용 (재학습 없이 저장된 모델 사용)
# =====================================================================
import torch
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from sklearn.preprocessing import RobustScaler
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

TICKER     = 'NVDA'
START_DATE = '2015-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')
PRED_DAYS  = 1
SAVE_DIR   = f'C:/Users/smhrd/Desktop/주식예측모델/models/{TICKER}'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 구조 (저장할 때랑 동일해야 함)
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, n_features, patch_len, d_model, dropout=0.1):
        super().__init__()
        self.patch_len  = patch_len
        self.projection = nn.Linear(patch_len * n_features, d_model)
        self.dropout    = nn.Dropout(dropout)
        self.norm       = nn.LayerNorm(d_model)
    def forward(self, x):
        B, L, F = x.shape
        n_patches = L // self.patch_len
        x = x[:, :n_patches * self.patch_len, :]
        x = x.reshape(B, n_patches, self.patch_len * F)
        x = self.projection(x)
        x = self.norm(x)
        return self.dropout(x)

class PatchTST(nn.Module):
    def __init__(self, n_features, seq_len, patch_len=10,
                 d_model=64, n_heads=4, n_layers=2, dropout=0.3):
        super().__init__()
        self.patch_embedding = PatchEmbedding(n_features, patch_len, d_model, dropout)
        n_patches = seq_len // patch_len
        self.pos_encoding = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model * n_patches, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x = self.patch_embedding(x)
        x = x + self.pos_encoding
        x = self.transformer(x)
        x = self.norm(x)
        x = x.flatten(1)
        return self.head(x)

# =====================================================================
# 📦 저장된 모델 & 메타 불러오기
# =====================================================================
with open(f'{SAVE_DIR}/meta.pkl', 'rb') as f:
    meta = pickle.load(f)

scaler       = meta['scaler']
selector     = meta['selector']
feature_cols = meta['feature_cols']
BEST_THRESHOLD = meta['BEST_THRESHOLD']
SEQ_LEN      = meta['SEQ_LEN']
W_LSTM       = meta['W_LSTM']
W_LGBM      = meta['W_LGBM']
W_RF         = meta['W_RF']

with open(f'{SAVE_DIR}/lgbm.pkl', 'rb') as f:
    lgbm_model = pickle.load(f)
with open(f'{SAVE_DIR}/rf.pkl', 'rb') as f:
    rf_model = pickle.load(f)

lstm_model = PatchTST(
    n_features = selector.transform(scaler.transform(np.zeros((1, len(feature_cols))))).shape[1],
    seq_len    = SEQ_LEN,
    patch_len  = meta['PATCH_LEN'],
    d_model    = meta['D_MODEL'],
    n_heads    = meta['N_HEADS'],
    n_layers   = meta['N_LAYERS'],
    dropout    = 0.3
).to(device)
lstm_model.load_state_dict(torch.load(f'{SAVE_DIR}/patchtst.pth', map_location=device))
lstm_model.eval()
print(f'✅ 모델 로드 완료! ({TICKER})')

# =====================================================================
# 📡 데이터 수집 & 피처 생성
# =====================================================================
def fetch_all_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open','High','Low','Close','Volume']]
    df.index = pd.to_datetime(df.index)

    tickers = {'SP500':'^GSPC','NDX':'^NDX','VIX':'^VIX',
                'DXY':'DX-Y.NYB','OIL':'CL=F','TNX':'^TNX'}
    ext = pd.DataFrame()
    for name, sym in tickers.items():
        try:
            data = yf.download(sym, start=start, end=end, progress=False)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            ext[name] = data['Close']
        except:
            pass
    return df.join(ext, how='left').ffill().dropna()

def add_features(d):
    if 'SP500' in d.columns: d['SP500_return_1d'] = d['SP500'].pct_change().shift(1)
    if 'NDX' in d.columns:   d['NDX_return_1d']   = d['NDX'].pct_change().shift(1)
    if 'DXY' in d.columns:   d['DXY_return_1d']   = d['DXY'].pct_change().shift(1)
    if 'OIL' in d.columns:   d['OIL_return_1d']   = d['OIL'].pct_change().shift(1)
    if 'VIX' in d.columns:   d['VIX_diff_1d']     = d['VIX'].diff().shift(1)
    if 'TNX' in d.columns:   d['TNX_diff_1d']     = d['TNX'].diff().shift(1)
    d['Return'] = d['Close'].pct_change()
    for i in [2,5,10,20]: d[f'Return_{i}d'] = d['Close'].pct_change(i)
    d['HL_ratio'] = (d['High']-d['Low'])/d['Close']
    d['OC_ratio'] = (d['Close']-d['Open'])/(d['Open']+1e-9)
    d['Gap'] = (d['Open']-d['Close'].shift(1))/(d['Close'].shift(1)+1e-9)
    for w in [5,10,20,60,120]:
        sma = SMAIndicator(d['Close'], window=w).sma_indicator()
        d[f'SMA_{w}'] = sma
        d[f'SMA_dist_{w}'] = (d['Close']-sma)/(sma+1e-9)
    d['EMA_12'] = EMAIndicator(d['Close'], window=12).ema_indicator()
    d['EMA_26'] = EMAIndicator(d['Close'], window=26).ema_indicator()
    macd = MACD(d['Close'])
    d['MACD'], d['MACD_signal'], d['MACD_diff'] = macd.macd(), macd.macd_signal(), macd.macd_diff()
    for w in [7,14,21]: d[f'RSI_{w}'] = RSIIndicator(d['Close'], window=w).rsi()
    d['Volume_ratio'] = d['Volume']/(d['Volume'].rolling(20).mean()+1e-9)
    d['Vol_5d']  = d['Return'].rolling(5).std()
    d['Vol_20d'] = d['Return'].rolling(20).std()
    d['Vol_ratio'] = d['Vol_5d']/(d['Vol_20d']+1e-9)
    if 'NDX_return_1d' in d.columns:
        d['TICKER_vs_NDX_1d'] = d['Return'].shift(1) - d['NDX_return_1d']
    raw_cols = ['SP500','NDX','VIX','DXY','OIL','TNX']
    d = d.drop(columns=[c for c in raw_cols if c in d.columns], errors='ignore')
    return d.ffill().bfill().dropna()

# =====================================================================
# 🔮 예측 실행
# =====================================================================
df_raw = fetch_all_data(TICKER, START_DATE, END_DATE)
df_f   = add_features(df_raw.copy())

X_new = np.where(np.isinf(df_f[feature_cols].values), np.nan, df_f[feature_cols].values)
X_new = pd.DataFrame(X_new, columns=feature_cols).ffill().bfill().values
X_new_sc  = scaler.transform(X_new)
X_new_sel = selector.transform(X_new_sc)
seq  = X_new_sel[-SEQ_LEN:][np.newaxis, ...]
flat = X_new_sel[-1:, :]

with torch.no_grad():
    lp = lstm_model(torch.tensor(seq, dtype=torch.float32).to(device)).cpu().numpy()[0][0]
gp = lgbm_model.predict_proba(flat)[0][1]
rp = rf_model.predict_proba(flat)[0][1]
prob = W_LSTM*lp + W_LGBM*gp + W_RF*rp

TARGET_THRESHOLD = BEST_THRESHOLD
signal_strength  = abs(prob - 0.5) * 200
HOLDING_DAYS     = 5

if prob > (0.5 + TARGET_THRESHOLD):
    decision    = "🔥 강력 매수 (STRONG LONG)" if signal_strength > 15 else "📈 조건부 매수 (LONG)"
    action_plan = f"오늘 종가에 진입 후 최소 {HOLDING_DAYS}영업일 동안 무조건 홀딩"
elif prob < (0.5 - TARGET_THRESHOLD):
    decision    = "🚨 위험 감지: 현금 대피 (CASH OUT)"
    action_plan = "신규 진입 절대 금지 및 기존 포지션 전량 현금화 후 관망"
else:
    decision    = "💤 방향성 불투명: 관망 (HOLD CASH)"
    action_plan = "매수 마진 미달. 현금 100% 쥐고 대기"

print("\n" + "="*65)
print(f" 🔮 NVDA 실전 매매 시그널 대시보드")
print("="*65)
print(f" 📅 기준 거래일  : {df_f.index[-1].strftime('%Y-%m-%d')} (종가: ${df_f['Close'].iloc[-1]:.2f})")
print(f" 🎯 타겟 예측    : 미래 {PRED_DAYS}영업일 뒤 방향성 (Threshold: {TARGET_THRESHOLD})")
print(f" 🛡️ 매매 룰      : 숏 절대 금지 | 강제 홀딩: {HOLDING_DAYS}일")
print("-"*65)
print(f" [모델 1] PatchTST : {lp*100:.1f}% ({'📈 BUY' if lp > 0.5 else '📉 CASH'})")
print(f" [모델 2] LightGBM : {gp*100:.1f}% ({'📈 BUY' if gp > 0.5 else '📉 CASH'})")
print(f" [모델 3] RF       : {rp*100:.1f}% ({'📈 BUY' if rp > 0.5 else '📉 CASH'})")
print("─"*65)
print(f" 🏆 최종 결론    : {decision}")
print(f" 📊 신뢰 강도    : {signal_strength:.1f}% (앙상블 확률: {prob*100:.1f}%)")
print(f" 🎯 매매 가이드  : {action_plan}")
print("="*65)

C:\Users\smhrd\AppData\Local\Temp\ipykernel_14888\1347959979.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


✅ 모델 로드 완료! (NVDA)


c:\Users\smhrd\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



 🔮 NVDA 실전 매매 시그널 대시보드
 📅 기준 거래일  : 2026-06-24 (종가: $199.00)
 🎯 타겟 예측    : 미래 1영업일 뒤 방향성 (Threshold: 0.001)
 🛡️ 매매 룰      : 숏 절대 금지 | 강제 홀딩: 5일
-----------------------------------------------------------------
 [모델 1] PatchTST : 49.6% (📉 CASH)
 [모델 2] LightGBM : 51.5% (📈 BUY)
 [모델 3] RF       : 50.6% (📈 BUY)
─────────────────────────────────────────────────────────────────
 🏆 최종 결론    : 📈 조건부 매수 (LONG)
 📊 신뢰 강도    : 1.2% (앙상블 확률: 50.6%)
 🎯 매매 가이드  : 오늘 종가에 진입 후 최소 5영업일 동안 무조건 홀딩
